# Studying nonconvex constellations via $\Delta\Phi(x,p)$

We apply $\Delta\Phi(x,p)$ and its visualizations to the study of a couple of Engelsma's 
examples of nonconvex constellations.  These are relatively long admissible constellations of low span
such that
$$ \pi(|s|) < {\rm length}(s).$$
These examples $s$ show that if the $k$-tuple conjecture is true, then the convexity conjecture
for $\pi(x)$ is false.  For such an $s$ we have
$$ \pi(\gamma_0+|s|) > \pi(\gamma_0) + \pi(|s|).$$

We analyze a counterexample $s$ with $(J,|s|)=(459,3242)$.
We start with a driving term for $s$ in ${\mathcal G}(11^\#)$ with initial generator $\gamma_0 = 1271$ and trace its evolution through subsequent stages of the sieve.

Guided by step R2 of the recursion $R: {\mathcal G}(p_{k-1}^\#) \longrightarrow {\mathcal G}(p_k^\#)$,
we use primorial coordinates or the primorial expansion for $\gamma_0(p_k)$ to track the incidences of $s$.
$$ \gamma_0(p_k) = \gamma_0 + m_1 \cdot 11^\# + m_2 \cdot 13^\# + \cdots + m_k \cdot p_{k-1}^\#$$
or 
$$ \gamma_0(p_k) = \gamma_0 + 11^\# (m_1 + 13(m_2 + 17(m_3 + \cdots +  p_{k-1}^\# \cdot m_k ))\cdots)$$
Each coefficient $m_i$ lies in the range $0 \le m_i < p_i$ and indicates which copy of ${\mathcal G}(p_{i-1}^\#)$ this image of $s$ lies in
under step R2 for ${\mathcal G}(p_{i-1}^\#)\longrightarrow {\mathcal G}(p_i^\#)$.

In [1]:
import pandas as pd
import numpy as np
import array
import itertools
from sympy import mod_inverse
import random

import matplotlib.pyplot as plt
from ipywidgets import interact
import ipywidgets as widgets
from IPython.display import display
plt.rcParams['figure.dpi'] = 300
plt.ion

import gc
import psutil
import sys
import csv
import pickle

In [2]:
# set up the array of small primes.  Start with primes19
primes19=np.load('primes19.npy')

In [3]:
primes19[0:20]

array([ 23,  29,  31,  37,  41,  43,  47,  53,  59,  61,  67,  71,  73,
        79,  83,  89,  97, 101, 103, 107])

In [4]:
# PRIMES:  The array smallp runs through primes from 2 up to 9699691
# 
smallp = np.concatenate(([2,3,5,7,11,13,17,19],primes19))
smallp[0:52], smallp[-5:]

(array([  2,   3,   5,   7,  11,  13,  17,  19,  23,  29,  31,  37,  41,
         43,  47,  53,  59,  61,  67,  71,  73,  79,  83,  89,  97, 101,
        103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167,
        173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229, 233, 239]),
 array([9699647, 9699649, 9699653, 9699667, 9699691]))

This notebook automates the analysis of Engelsma $(J,|s|)$-counterexamples to the convexity conjecture 
for constellations among primes.
We use $\Delta \Phi$ to visualize the constellation.  
The array DelPhi contains the lower points on the vertical segments for $\Delta \Phi(x,p)$.  The upper points on these segments are (DelPhi+1)

In [5]:
# block to check the available system memory
gc.collect()
memory = psutil.virtual_memory()
available_memory = memory.available
del memory
print(f"Available memory: {available_memory / (1024 ** 2):.2f} MB")

Available memory: 3717.89 MB


## Engelsma counterexamples to the convexity conjecture
Engelsma et al. have identified several examples of admissible constellations for which their lengths $J$ exceed the number of primes under their span
$$ \pi(|s|) < J$$
The shortest counterexamples $s$ have length $J=458$ and span $|s|=3240$.  These counterexamples can be extended by a single gap $2$ to produce a second admissible counterexample of length $J=459$ and span $|s|=3242$.

These constellations have unique driving terms in the cycle ${\mathcal G}(11^\#)$ with various $\gamma_0$.  
This driving term has a unique image through the next several stages of the sieve.

In [6]:
# Engelsma counterexample of 458 gaps of span 3240.
EngelsmaL = np.array([2,4,2,4,8,6,4,2,10,6,2,6,12,4,6,12,2,4,2,4,8,6,12,4,6,8,6,4,2,4,14,10,12,
                     2,10,2,4,12,2,10,2,4,6,8,6,6,6,4,6,12,6,2,4,8,6,10,2,4,8,16,6,6,2,6,10,2,22,
                     2,6,4,6,2,10,12,8,6,6,6,4,6,8,4,2,4,2,18,10,2,10,14,4,14,10,2,4,12,2,18,10,2,
                     6,6,10,18,2,10,6,8,6,4,14,6,10,6,6,8,6,10,6,8,4,2,6,10,12,6,12,2,6,4,2,4,6,6,2,
                     10,12,14,4,8,4,8,6,4,6,2,12,6,6,10,6,12,2,4,14,12,4,2,4,6,12,2,4,12,12,2,6,4,20,
                     4,2,18,4,6,2,10,2,6,6,10,8,16,2,12,10,2,4,6,6,12,6,6,6,20,4,14,4,2,4,14,6,16,8,
                     6,10,2,10,2,6,12,10,12,6,2,10,8,4,2,18,10,2,6,4,18,6,2,22,12,6,2,16,6,6,2,6,6,4,
                     14,10,2,10,14,6,4,6,6,2,18,10,8,4,2,18,6,4,6,6,6,8,6,4,2,10,2,12,4,2,10,2,12,4,2,
                     4,8,6,10,6,6,2,6,16,18,2,6,4,8,6,10,6,6,8,10,2,12,6,4,2,4,2,10,12,2,6,4,8,10,2,16,
                     30,8,4,2,6,4,8,16,14,12,4,2,10,12,6,24,2,4,2,18,6,10,6,2,6,6,16,8,4,6,2,4,18,8,
                     4,8,6,6,6,10,8,4,2,4,12,2,12,6,10,14,10,8,6,4,2,10,6,8,10,2,4,8,10,8,4,6,2,6,10,
                     6,8,6,10,2,10,2,12,4,6,8,6,4,2,16,8,4,8,10,12,2,4,2,22,6,2,4,6,14,6,4,8,10,2,6,10,
                     2,10,12,6,8,4,2,10,8,6,6,16,18,6,8,6,4,6,6,8,6,6,6,4,6,2,12,4,2,10,14,6,10,2,6,10,
                     6,6,2,10,2,6,4,14,4,2,34], dtype=int)
EngelsmaR = np.array([4,2,4,8,6,4,2,10,6,2,6,12,4,6,12,2,4,2,4,8,6,12,4,6,8,6,4,2,4,14,10,12,
                     2,10,2,4,12,2,10,2,4,6,8,6,6,6,4,6,12,6,2,4,8,6,10,2,4,8,16,6,6,2,6,10,2,22,
                     2,6,4,6,2,10,12,8,6,6,6,4,6,8,4,2,4,2,18,10,2,10,14,4,14,10,2,4,12,2,18,10,2,
                     6,6,10,18,2,10,6,8,6,4,14,6,10,6,6,8,6,10,6,8,4,2,6,10,12,6,12,2,6,4,2,4,6,6,2,
                     10,12,14,4,8,4,8,6,4,6,2,12,6,6,10,6,12,2,4,14,12,4,2,4,6,12,2,4,12,12,2,6,4,20,
                     4,2,18,4,6,2,10,2,6,6,10,8,16,2,12,10,2,4,6,6,12,6,6,6,20,4,14,4,2,4,14,6,16,8,
                     6,10,2,10,2,6,12,10,12,6,2,10,8,4,2,18,10,2,6,4,18,6,2,22,12,6,2,16,6,6,2,6,6,4,
                     14,10,2,10,14,6,4,6,6,2,18,10,8,4,2,18,6,4,6,6,6,8,6,4,2,10,2,12,4,2,10,2,12,4,2,
                     4,8,6,10,6,6,2,6,16,18,2,6,4,8,6,10,6,6,8,10,2,12,6,4,2,4,2,10,12,2,6,4,8,10,2,16,
                     30,8,4,2,6,4,8,16,14,12,4,2,10,12,6,24,2,4,2,18,6,10,6,2,6,6,16,8,4,6,2,4,18,8,
                     4,8,6,6,6,10,8,4,2,4,12,2,12,6,10,14,10,8,6,4,2,10,6,8,10,2,4,8,10,8,4,6,2,6,10,
                     6,8,6,10,2,10,2,12,4,6,8,6,4,2,16,8,4,8,10,12,2,4,2,22,6,2,4,6,14,6,4,8,10,2,6,10,
                     2,10,12,6,8,4,2,10,8,6,6,16,18,6,8,6,4,6,6,8,6,6,6,4,6,2,12,4,2,10,14,6,10,2,6,10,
                     6,6,2,10,2,6,4,14,4,2], dtype=int)

In [7]:
# This function returns a list of available residues mod inp that begin admissible images of constellation incons
def admissible(inp, incons):
    # calculate list of covered residues mod p by the input constellation
    rez=0
    covered_rez={0}
    i=0
    while (i < len(incons)):
        rez = (rez + incons[i])% inp
        if rez not in covered_rez:
            covered_rez.add(rez)
        i += 1
    # are all residues covered?
    n_available_rez = inp - len(covered_rez)
    i=1
    available_rez = set()
    # each entry in covered_rez corresponds to a starting residue of (inp-rez)mod inp
    while (i < inp):
        test_rez = inp - i
        if test_rez not in covered_rez:
            available_rez.add(i)
        i += 1
    return available_rez

In [8]:
# this function calculates the value of mk such that 
#  0 <= mk < pk  and  mk*pml(p_{k-1}) + r0 = rk mod pk
# reminder for indexing that pk = smallp[k+4], a shift of 4, and p0=11
def primorialm(k,rk,r0):
    pk = smallp[k+4]
    i = 0
    pmlp = 1
    while (i < k+4):
        pmlp = (pmlp * smallp[i]) % pk
        i += 1
    mk = (mod_inverse(pmlp, pk) * (rk-r0) ) % pk
    return mk

In [9]:
# This function returns the generator in G(11#) that begins a driving term for the input constellation
def findgamma11(incons):
    i=1
    gamma0 = 1
    pmlp = 2
    
    while (i <= 4):  # smallp[4]=11
        pk = int(smallp[i])
        rez = admissible(pk, incons)

        if (len(rez) != 1):   # We assume that the generator in G(11#) is unique
            print(f"UNEXPECTED: p {pk} rez {len(rez)} {rez}")

        target_r = int(list(rez)[0])
        r0 = gamma0 % pk
        mk = (mod_inverse(pmlp, pk) * (target_r-r0)) % pk

        gamma0 += mk*pmlp
        pmlp = (pmlp * smallp[i])

        i += 1

    return gamma0        


In [10]:
gamma0 = findgamma11(EngelsmaL)
gamma0

np.int64(1271)

In [11]:
# Read in constellations from text file
Eng459_constellations = np.loadtxt('tuples_460_3242.txt', dtype=int)
Eng459_constellations.shape, Eng459_constellations.shape[0]
# Save as numpy file


((58, 459), 58)

In [12]:
#
Eng459_constellations[0:30,0:12]

array([[ 2,  4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6],
       [ 2,  4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6],
       [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
       [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
       [ 2,  4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6],
       [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
       [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
       [ 2,  4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6],
       [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
       [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
       [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
       [ 2,  4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6],
       [ 2,  4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6],
       [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
       [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
       [ 2,  4,  2,  4,  8,  6,  4,  2, 10,  6,  2,  6],
       [ 2,  4, 14,  4,  6,  2, 10,  2,  6,  6, 10,  6],
       [ 2,  4, 14,  4,  6,  2,

In [14]:
i = 0
while (i < 58):
    const_k = Eng459_constellations[i]
    gamma0 = findgamma11(const_k)
    print(f"{i} {len(const_k)} {np.sum(const_k)} gamma0 {gamma0}")
    i += 1

0 459 3242 gamma0 107
1 459 3242 gamma0 107
2 459 3242 gamma0 1271
3 459 3242 gamma0 1271
4 459 3242 gamma0 107
5 459 3242 gamma0 1271
6 459 3242 gamma0 1271
7 459 3242 gamma0 107
8 459 3242 gamma0 1271
9 459 3242 gamma0 1271
10 459 3242 gamma0 1271
11 459 3242 gamma0 107
12 459 3242 gamma0 107
13 459 3242 gamma0 1271
14 459 3242 gamma0 1271
15 459 3242 gamma0 1271
16 459 3242 gamma0 107
17 459 3242 gamma0 107
18 459 3242 gamma0 1271
19 459 3242 gamma0 107
20 459 3242 gamma0 107
21 459 3242 gamma0 1271
22 459 3242 gamma0 1271
23 459 3242 gamma0 1271
24 459 3242 gamma0 107
25 459 3242 gamma0 107
26 459 3242 gamma0 107
27 459 3242 gamma0 107
28 459 3242 gamma0 1271
29 459 3242 gamma0 107
30 459 3242 gamma0 1271
31 459 3242 gamma0 107
32 459 3242 gamma0 107
33 459 3242 gamma0 1271
34 459 3242 gamma0 1271
35 459 3242 gamma0 1271
36 459 3242 gamma0 107
37 459 3242 gamma0 107
38 459 3242 gamma0 1271
39 459 3242 gamma0 1271
40 459 3242 gamma0 107
41 459 3242 gamma0 1271
42 459 3242 gamma0 127

## Main loop through Engelsma (459,3242)-counterexamples
What follows is the main loop through the 58 counterexamples identified by Thomas Engelsma for the case $(J,|s|)=(459,3242)$.
The first blocks calculate the unique prefix for each counterexample's primorial coordinates 
and sort the counterexamples by their prefixes.

Then for each constellation we search for primorial expansions that have sequences of 0's.

In [15]:
# ==================================================
# MAIN LOOP through COUNTEREXAMPLES ================
# =========================================================
#  The featured counterexample is Eng_459constellations[14]
# =========================================================
debug_verbose = False
num_cons = Eng459_constellations.shape[0]

Eng459_prefixes = []

# For each constellation --
icons = 0

while (icons < num_cons):
    constellation_k = Eng459_constellations[icons]
    gammam_list = []
    
    # Calculate gamma0 in G(11#)
    gamma0 = findgamma11(constellation_k)

    # Calculate residues across ranges of primes p
    # For each prime >= 11, so index i >= 4, we record the list of admissible residues for gamma_0 mod p[i]
    rezlist = []
    i=4
    while (smallp[i] < 250):
        p = smallp[i]
        rezp = admissible(p,constellation_k)
        rezlist.append(rezp)
        i += 1

    # Summarize num_admissible across the primes p
    num_admissible = [len(rezlist[i]) for i in range(len(rezlist))]
    num_admissible = np.array(num_admissible)

    if debug_verbose:
        print(f"{icons:2d} gamma0 {gamma0:4d} num_adm {num_admissible[0:60]}")
        for element in rezlist:
            print(f" {list(element)[0]}", end=' ')
        print()

    # Calculate primorial coordinates for unique prefix
    ip = 1
    gamma_m = [int(gamma0)]

    while (num_admissible[ip] == 1):
        pk = smallp[ip+4]  # primes are offset 4 in array, p0=11 so we start at pk=13
        
        # calculate the residue r0 mod pk
        r0 = gamma_m[0] % pk
        i = 1
        rpml = 2310 % pk  # 11# mod pk
        while (i < ip):
            r0 = (r0 + gamma_m[i]*rpml) % pk
            rpml = (rpml * smallp[i+4]) % pk
            i += 1

        target_r = list(rezlist[ip])[0]
        mk = primorialm(ip,target_r,r0)
        gamma_m.append(int(mk))
        ip += 1  # next prime

    
    # Report and save this information
    Eng459_prefixes.append(gamma_m)

    print(f"{icons:2d} prefix {Eng459_prefixes[icons]}")
    icons += 1   # next constellation - 


 0 prefix [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 13, 39, 42, 17, 23, 21, 20, 83, 22, 9, 81, 107, 103, 8, 53]
 1 prefix [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 13, 39, 42, 17, 23, 21, 20, 49, 64, 50, 10, 29, 28, 27, 38]
 2 prefix [1271, 5, 8, 9, 17, 21, 29, 13, 2, 8, 0, 32, 45, 47, 27, 28, 55, 55, 61, 68, 47, 36, 52, 96, 79, 84, 99, 92]
 3 prefix [1271, 5, 8, 9, 17, 21, 29, 13, 2, 8, 0, 32, 45, 47, 27, 28, 55, 55, 61, 70, 23, 51, 75, 2, 86, 23, 71, 104]
 4 prefix [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 13, 39, 42, 17, 23, 21, 18, 73, 49, 27, 104, 22, 89, 55, 26]
 5 prefix [1271, 5, 8, 9, 17, 21, 29, 13, 2, 8, 0, 32, 45, 47, 27, 28, 55, 55, 61, 70, 52, 33, 55, 75, 37, 98, 107, 74]
 6 prefix [1271, 5, 8, 9, 17, 21, 29, 13, 2, 8, 0, 32, 45, 47, 27, 28, 55, 55, 61, 68, 76, 18, 32, 62, 31, 46, 9, 63]
 7 prefix [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 13, 39, 42, 17, 23, 21, 20, 20, 82, 70, 44, 77, 66, 117, 67]
 8 prefix [1271, 5, 8, 9, 17, 21, 29, 13, 2, 8, 0,

In [16]:
Eng_pre_sorted, Eng_consts = zip(*sorted(zip(Eng459_prefixes, Eng459_constellations)))

In [17]:
i=0
while (i < len(Eng_pre_sorted)):
    print(f"{i:2d} {len(Eng_pre_sorted[i])} m {Eng_pre_sorted[i]}")
    print(f" {Eng_consts[i][0:24]}")
    i += 1

 0 29 m [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 4, 4, 53, 64, 11, 39, 27, 17, 44, 1, 78, 22, 108, 29, 112, 72]
 [ 2  4 14  4  6  2 10  2  6  6 10  6  2 10  6  2 12 10  2  4 12  2  6  4]
 1 29 m [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 5, 60, 51, 70, 42, 25, 33, 22, 73, 67, 66, 95, 17, 109, 111, 99]
 [ 2  4 14  4  6  2 10  8  6 10  6  2 10  6  2 12 10  2  4 12  2  6  4  6]
 2 29 m [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 5, 60, 51, 70, 42, 25, 33, 35, 51, 12, 18, 55, 43, 42, 44, 132]
 [ 2  4 14  4  6  2 10  8  6 10  6  2 10  6  2 12 10  2  4 12  2  6  4  6]
 3 29 m [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 5, 60, 51, 70, 42, 25, 33, 37, 1, 4, 60, 40, 21, 71, 114, 10]
 [ 2  4 14  4  6  2 10  8  6 10  6  2 10  6  2 12 10  2  4 12  2  6  4  6]
 4 29 m [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 5, 60, 51, 70, 42, 25, 33, 50, 80, 51, 11, 0, 47, 4, 47, 43]
 [ 2  4 14  4  6  2 10  8  6 10  6  2 10  6  2 12 10  2  4 12  8  4  6  6]
 5 29 m [107, 6, 8, 9, 5, 7, 1

## NOTE: Sorted by primorial coordinates
The constellations and their primorial prefixes are now sorted by their unique prefixes of primorial coordinates.
This data is saved to file so that we can start further explorations from here.

In [18]:
'''  Commenting out this block to protect the existing files
with open('Eng459_sorted.csv','w',newline='\n') as fp:
    writer = csv.writer(fp)
    writer.writerows(Eng_consts)

with open('Eng459_prefixes.csv','w',newline='\n') as fq:
    writerB = csv.writer(fq)
    writerB.writerows(Eng_pre_sorted)
'''

"  Commenting out this block to protect the existing files\nwith open('Eng459_sorted.csv','w',newline='\n') as fp:\n    writer = csv.writer(fp)\n    writer.writerows(Eng_consts)\n\nwith open('Eng459_prefixes.csv','w',newline='\n') as fq:\n    writerB = csv.writer(fq)\n    writerB.writerows(Eng_pre_sorted)\n"

In [19]:
smallp[0:10],smallp[450:463]

(array([ 2,  3,  5,  7, 11, 13, 17, 19, 23, 29]),
 array([3187, 3191, 3203, 3209, 3217, 3221, 3229, 3251, 3253, 3257, 3259,
        3271, 3299]))

## $\Delta \Phi$ to picture $p$-rough numbers

The count of $p$-rough numbers up through $x$ is denoted $\Phi(x,p)$.
The graph of $\Phi(x,p)$ has a line of symmetry
$$ \tilde{\Phi} = \frac{\phi(p^\#)}{p^\#} x = \frac{1}{\mu} x $$
where $\mu = \frac{p^\#}{\phi(p^\#)}$ is the mean size of the gaps in
$\mathcal{G}(p^\#)$.

So we work with $\Delta \Phi(x,p)$, which measures the deviation of $\Phi(x,p)$ around its line of symmetry.
$$ \Delta \Phi(x,p) = \Phi(x,p) - \frac{1}{\mu} x$$

The deviations of the $p$-rough numbers from their line of symmetry
$$\tilde{\Phi} = \frac{1}{\mu}x$$ 
are periodic and bounded.  Thus the behavior of $\Phi(x,p)$ is completely described by the behavior of $\Delta \Phi(x,p)$
over the first cycle $\mathcal{G}(p^\#)$, or using the rotational symmetry around $x=1+\frac{p^\#}{2}$ over the first
half of this cycle.

$\Delta \Phi(x,p)$ provides a good visualization of the $p$-rough numbers.

To use this visualization here, we use the average prime gap over the first $462$ primes, $p_{462}=3271$

| $k$ | $456$ | $457$ | $458$ | $459$ | $460$ | $461$ | $462$ | $463$ |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| $p_k$ | $3221$ | $3229$ | $3251$ | $3253$ | $3257$ | $3259$ | $3271$ | $3299$ |
| $\max|s|$ |  | $3236$ | $3240$ | $3242$ | $3276$ | | | |



In [20]:
# average gap size for 
mu_gap = 3271/462
mu_recip = -1/mu_gap
print("mu",mu_gap, "neg reciprocal (slope)", mu_recip)

mu 7.08008658008658 neg reciprocal (slope) -0.14124121063894834


In [21]:
# create the constellation of prime gaps
prime_constellation = smallp[1:462]-smallp[0:461]
prime_constellation = np.concatenate(([2], prime_constellation))


In [22]:
# this function interleaves arr1 with arr2, which we need for plotting the vertical segments
def interleave_np(arr1, arr2):
    stacked_arr = np.stack((arr1, arr2),axis=1)
    return stacked_arr.flatten().tolist()

In [23]:
# Plotting pi function vs Engelsma constellation (459,3242) 
# Interleaving data to get the stepped graph, rendering the vertical segments
# 
def Eng459plot(input_dex):
    input_s = Eng_consts[input_dex][1:]

    # data for plotting the prime constellation
    xp = smallp[0:462] # 462 values from 2 to 3271
    xp = interleave_np(xp,xp)  # ... doubled up...
    xp = np.concatenate(([0],xp)) # 1+2*462 values from 0 to 3271

    delpi = np.zeros(462) # we will calculate the lower points first
    i=1
    delpi[0] = 2*mu_recip
    while (i < 462):
        delpi[i] = delpi[i-1] + 1 + prime_constellation[i]*mu_recip
        i += 1

    delpi = interleave_np(delpi, (delpi + 1)) # interleave the lower points and upper points for the vertical segments
    delpi = np.concatenate(([0],delpi))

    pidf = pd.DataFrame({'x':xp, 'picnt': delpi})

    # data for plotting the input constellation
    xcons = np.cumsum(input_s)
    xcons = interleave_np(xcons, xcons)
    xcons = np.concatenate(([0],xcons))

    delconst = np.zeros(len(input_s))
    delconst[0] = input_s[0]* mu_recip
    i=1
    while (i < len(input_s)):
        delconst[i] = delconst[i-1] + 1 + input_s[i]*mu_recip
        i += 1

    delconst = interleave_np(delconst, (delconst+1))
    delconst = np.concatenate(([0],delconst))

    data_sample = {'x': xcons, 'DelPhi': delconst}
    df = pd.DataFrame(data_sample)

    # plotting the two curves
    plt.clf()
    fig, ax = plt.subplots()
    fig.set_size_inches(14,9)
    ax.set_title(f"Engelsma(459, 3242) #{input_dex} vs $\pi(n)$ as segments of $\Delta \Phi(x,\mu)$")
    ax.grid(axis='y', color='#080408', lw=0.125 )
     
    ax.plot(xcons, delconst, color='#2222AF', lw=0.125, label='DelPhi')
    ax.plot(xp, delpi, color='#AF0000', lw=0.25, label='pi(n)')
# ax.set_ylim(-8,12)

    plt.show()

xEng459Select = widgets.IntSlider(value=29, min=0, max=57, description="Which (459,3242)", 
                                  layout=widgets.Layout(width='80%'), style={'description_width':'90pt'}, disabled=False)

interact(Eng459plot, input_dex=xEng459Select)

interactive(children=(IntSlider(value=29, description='Which (459,3242)', layout=Layout(width='80%'), max=57, …

<function __main__.Eng459plot(input_dex)>

## Explore the constellations one at a time
We look at each of the 58 $(459,3242)$-counterexamples.  We have the constellation, its numbers of admissible residues across
several primes, and the corresponding unique primorial coordinates starting in ${\mathcal G}(11^\#)$.

For each constellation we find all of the extensions to its primorial coordinates through the next several primes.  This exhaustive search
is limited by the sheer number of possibilities.  We then take an opportunistic random approach to searching the larger space of
extensions.  We are looking for long sequences of coordinates $m_k=0$.

In [24]:
len(Eng_pre_sorted[5]), Eng_pre_sorted[5][:]

(29,
 [107,
  6,
  8,
  9,
  5,
  7,
  1,
  23,
  38,
  34,
  46,
  20,
  13,
  5,
  60,
  51,
  70,
  42,
  25,
  33,
  71,
  60,
  65,
  23,
  10,
  97,
  52,
  129,
  0])

In [49]:
Eng459_prefixesB = []
with open('Eng459_prefixes.csv', 'r') as fqtr:
    pref_reader = csv.reader(fqtr)
    for row in pref_reader:
        Eng459_prefixesB.append([int(x) for x in row])
fqtr.close()

In [53]:
len(Eng459_prefixesB[30])

28

In [51]:
Eng459_sortedB = []
with open('Eng459_sorted.csv', 'r') as fqtr:
    ssort_reader = csv.reader(fqtr)
    for row in ssort_reader:
        Eng459_sortedB.append([int(x) for x in row])
fqtr.close()

In [57]:
np.log10(len(Eng459_sortedB[30]))

np.float64(2.661812685537261)

In [87]:
# We start with the unique prefixes in Eng_pre_sorted and the constellations in Eng_const
# The array smallp starts at p=2, and p0 for the prefixes is p=11.  The primorial coordinate m[k] corresponds to smallp[k+4]

debug_verbose = True
saveto_file = False     # flag for writing the _results and _mzeros files
ic = int(0)  # index for the constellation

while (ic < len(Eng459_sortedB)):
    pref_len = len(Eng459_prefixesB[ic])  # for the ic-th counterexample - 
    Eng459s = Eng459_sortedB[ic]             # get the constellation
    mprefix = Eng459_prefixesB[ic]        # get the unique prefix for the primorial coordinates

    # for this constellation and prefix, determine the admissible residues for an exhaustive search 
    #  of up to 20M examples each
    rezlist = []
    ip0 = 4 + pref_len
    while (smallp[ip0] < 5000): 
        p = smallp[ip0]
        rezp = admissible(p,Eng459s)
        rezlist.append(rezp)
        ip0 += 1

    # Summarize num_admissible across the primes p
    num_admissible = [len(rezlist[i]) for i in range(len(rezlist))]
    num_admissible = np.array(num_admissible)

    if debug_verbose:
        print(f"{ic:2d} gamma0 {Eng459s[0]:4d} len_prefix {pref_len:2d} num_adm {num_admissible[0:60]}")
        # for element in rezlist:
            # print(f" {list(element)[0]}", end=' ')
        # print()

    # We extend the primorial coordinates 
    num_instances = 1
    m_ext_list = []
    k = 0      # index for pk beyond the prefix, e.g. for num_admissible and rezlist

    
    if debug_verbose:
        print(f"k {k} num_admissible {num_admissible[k]} total instances {num_instances}", end='\r')

    # initiate the extensions in first iteration on k
    ik = k+pref_len   # index for smallp[] corresponding to k
    pk = smallp[ik+4]
    num_instances = num_admissible[k]
    i = 0
    while (i < num_admissible[k]):
        m_ext_list.append(list(mprefix))

        target_residue = list(rezlist[k])[i]
        # calculate residue r0 modulo pk
        # set initial conditions for this loop
        j = 1
        r0 = m_ext_list[i][0] % pk
        rpml = 2310 % pk  # initialize this factor at 11#
        while (j < ik):  # calculate the residue for the primorial expansion
            r0 = (r0 + m_ext_list[i][j] * rpml) % pk
            rpml = (rpml * smallp[j+4]) % pk
            j += 1
        # calculate next primorial coefficient mk
        mk = primorialm(ik,target_residue,r0)
        (m_ext_list[i]).append(int(mk))

        if debug_verbose:
            print(f"k {k} {ik} p {smallp[ik+4]} i {i} targetr {target_residue} r0 {r0}")
            for element in m_ext_list[i]:
                print(f"{element:3d}", end=" ")
            print()

        i += 1

    # now iterate k through the primes pk.  
    #  k is the index in num_admissible[] which starts after the unique prefix
    #  ik is the corresponding index in smallp[], as an offset from 11
    k += 1
    ik += 1

    while (num_instances < 4000000):   
        num_instances *= num_admissible[k]
        pk = smallp[ik+4]
    
        iprefix=0
        num_prefixes = len(m_ext_list)
        # iprefix is the index through existing primorial expansions
        # i is the index through admissible residues at this stage
        
        while (iprefix < num_prefixes):
            # calculate r0 mod pk
            j = 1
            r0 = m_ext_list[iprefix][0] % pk
            rpml = 2310 % pk  # initialize this factor at 11#
            while (j < ik):  # the expansion for r 
                r0 = (r0 + m_ext_list[iprefix][j] * rpml) % pk
                rpml = (rpml * smallp[j+4]) % pk
                j += 1

            # extend the first copy in place
            # save the prefix
            m_start = list(m_ext_list[iprefix].copy())
            # calculate next primorial coefficient mk
            target_residue = list(rezlist[k])[0]
            mk = primorialm(ik,target_residue,r0)
            (m_ext_list[iprefix]).append(int(mk))

            if ((iprefix % 4096)==0):
                print(f"ic {ic} k {k} {ik} p {pk} i {iprefix} targetr {target_residue} r0 {r0} length {len(m_ext_list[iprefix])}", end='\r')
        
            i=1
            while (i < num_admissible[k]):
                icopy = len(m_ext_list)
                m_ext_list.append(m_start.copy())  # icopy is the index for this copy
                target_residue = list(rezlist[k])[i]
                mk = primorialm(ik,target_residue,r0)
                (m_ext_list[icopy]).append(int(mk))
                # print(f"Copy {icopy} k {k} p {smallp[k+4]} i {iprefix} {i} targetr {target_residue} r0 {r0} length {len(gammam_list[icopy])}")

                i += 1

            iprefix += 1  # next prefix

        k += 1     # next prime - depth of search
        ik += 1

    # Process and record the results for this constellation
    filename = 'Eng459_' + str(ic) + '_results.csv'
    k -= 1
    ik -= 1
    
    # record ic, prefix_len, p0, k, pk, num_instances, num_admissible
    # XXXQHERE [6/22] - how much of num_admissible to save?
    if saveto_file:
        results_list = [ic, pref_len, smallp[4+pref_len], k, smallp[k+pref_len+4], num_instances, num_admissible]
    
        # Factors of asymptotic relative population 
        # - up through J+1=460, and up through |s|/2 = 1621
        # We calculate the log of this factor, anticipating floating point overflow
        ws_J_log = 0
        i = 0    # indexing here: admissible 0 == primes 4+pref_len
        pk = smallp[i + 4 + pref_len]
        while (pk <= 460):
            ws_J_log += np.log10(num_admissible[i])
            i += 1
            pk = smallp[i + 4 + pref_len]

        ws_s_log = 0
        while (pk <= 1621):
            ws_s_log += np.log10(num_admissible[i] / (pk-460))
            i += 1
            pk = smallp[i + 4 + pref_len]

        results_list.append([ws_J_log, ws_s_log])

        with open(filename, 'w', newline='\n') as fptr:
            writer = csv.writer(fptr)
            writer.writerow(results_list)
        fptr.close()
    
    # Identify instances whose primorial coordinates end in sequences of 0's
    num_mext = len(m_ext_list)
    zero_list = []
    imext = 0
    while (imext < num_mext):
        j = len(m_ext_list[imext])-1
        while ( m_ext_list[imext][j] == 0):
            j -= 1
        nzer = len(m_ext_list[imext]) - 1 - j
        if (nzer > 0):
            zero_list.append([nzer,imext])
        imext += 1

    if saveto_file:
        if (len(zero_list)>0):
            print(f"Saving {len(zero_list)} extensions out of {num_mext}")
            sorted_zero_list = sorted(zero_list, reverse=True)
            max_zeros = sorted_zero_list[0][0]
            m_zeros = [] 
            j=0
            while (j < len(sorted_zero_list)):
                imext = sorted_zero_list[j][1]
                entry = [ sorted_zero_list[j], m_ext_list[imext]]
                m_zeros.append(entry)
                j += 1

            filename = 'Eng459_' + str(ic) + '_mzeros.csv'

            with open(filename, 'w', newline='\n') as fptr:
                writer = csv.writer(fptr)
                writer.writerows(m_zeros)
            fptr.close()


    ic +=1   # Loop into next constellation (459,3242)



 0 gamma0    2 len_prefix 29 num_adm [  2   1   1   3   1   2   3   4   3   9   5   5   5  10   8  12   7   9
   9  18  14  17  25  21  17  22  27  21  28  31  37  34  39  38  50  52
  48  54  59  64  72  68  73  80  89  85  87  94  95 103 103 102 108 112
 112 115 132 126 134 129]
k 0 29 p 139 i 0 targetr 24 r0 91ces 1
107   6   8   9   5   7   1  23  38  34  46  20  13   4   4  53  64  11  39  27  17  44   1  78  22 108  29 112  72 132 
k 0 29 p 139 i 1 targetr 90 r0 91
107   6   8   9   5   7   1  23  38  34  46  20  13   4   4  53  64  11  39  27  17  44   1  78  22 108  29 112  72 114 
Saving 22910 extensions out of 48600002 r0 6 length 4343
 1 gamma0    2 len_prefix 29 num_adm [  2   1   1   3   2   3   2   4   3   9   5   5   6  11   8  12   8   9
  10  17  16  16  24  19  18  21  27  21  29  32  38  35  39  39  48  49
  49  56  60  66  69  65  73  82  88  81  89  94  96 104  99 102 109 113
 110 115 129 123 133 130]
k 0 29 p 139 i 0 targetr 24 r0 136es 1
107   6   8   9   5   7  

## Data files _results and _mzeros
The search through primorial coordinates above produces two sets of output files:  <i>Eng459_xx_results.csv</i> and <i>Eng459_xx_mzeros.csv</i>.  At this point, the search is breadth-first and exhaustive.  So not very deep.  

In order to pursue the survivial of any incidence of these (459,3242)-counterexamples, we have to switch to a greedy depth-first
approach.  Survival is indicated by a <i>long</i> sequence of consecutive primorial coordinates $m_k = 0$.  So we start by considering
those constellations with primorial expansions from the current search that end with at least one $m_k=0$.

<i>Eng459_xx_results.csv</i> contains summary notes about the search over the (459,3242)-counterexample of index <i>xx</i> in the file 
<i>Eng459_sorted.csv</i>  The first few fields in the <i>_results</i> file are the index <i>xx</i> of the counterexample; the length of
the unique prefix of the primorial coordinates, starting at $p_{0}=11$; the prime $p_{k_0}$ just beyond this unique prefix, where there is more than one admissible residue; the depth $k$ of the breadth-first search into extensions of the prefix; the prime corresponding to 
the last $m_k$ recorded; and the number of admissible instances searched.  Then the array of the numbers of admissible instances $\bmod p$
is listed, starting at $p_{k_0}$.  After this array we list the two factors (in log-base-10) of the asymptotic relative population
for this constellation:
$$ w_{s,J}(\infty) \; = \; \prod_{p \le J+1} (p - \nu(p)) \cdot \prod_{p > J+1} \frac{p - \nu(p)}{p - J-1}$$

<i>Eng459_xx_mzeros.csv</i> contains data about the extensions of the primorial coordinates for the (459,3242)-counterexample of
index <i>xx</i> in the file <i>Eng459_sorted.csv</i>  Each row of data starts with a two-element array:  
the number of terminal zeroes for this extension, and the index of the extension in the breadth-first search.  
This is followed by the primorial coordinates from $p_0=11$.

## Counterexample surviving the sieve
The Engelsma counterexample (459,3242) has a unique image up through ${\mathcal G}(131^\#)$.  The constellation itself first appears in 
${\mathcal G}(113^\#)$, and there are no longer driving terms.

By ${\mathcal G}(457^\#)$ there are $2.10278720 \; E73$ images of this constellation.
Over $461 \le p_k \le 1619$ the relative population is 
$$\prod_{461}^{1619} \frac{q-\nu(q)}{q-460} \; = \; 2.51736042 \;E17$$

Thus the asymptotic relative population of the Engelsma counterexample (459,3242) is
$$w_{s,459}(\infty) = 5.29347327 \cdot E90$$

In [38]:
del_smallp[0:459]

array([ 2,  1,  2,  2,  4,  2,  4,  2,  4,  6,  2,  6,  4,  2,  4,  6,  6,
        2,  6,  4,  2,  6,  4,  6,  8,  4,  2,  4,  2,  4, 14,  4,  6,  2,
       10,  2,  6,  6,  4,  6,  6,  2, 10,  2,  4,  2, 12, 12,  4,  2,  4,
        6,  2, 10,  6,  6,  6,  2,  6,  4,  2, 10, 14,  4,  2,  4, 14,  6,
       10,  2,  4,  6,  8,  6,  6,  4,  6,  8,  4,  8, 10,  2, 10,  2,  6,
        4,  6,  8,  4,  2,  4, 12,  8,  4,  8,  4,  6, 12,  2, 18,  6, 10,
        6,  6,  2,  6, 10,  6,  6,  2,  6,  6,  4,  2, 12, 10,  2,  4,  6,
        6,  2, 12,  4,  6,  8, 10,  8, 10,  8,  6,  6,  4,  8,  6,  4,  8,
        4, 14, 10, 12,  2, 10,  2,  4,  2, 10, 14,  4,  2,  4, 14,  4,  2,
        4, 20,  4,  8, 10,  8,  4,  6,  6, 14,  4,  6,  6,  8,  6, 12,  4,
        6,  2, 10,  2,  6, 10,  2, 10,  2,  6, 18,  4,  2,  4,  6,  6,  8,
        6,  6, 22,  2, 10,  8, 10,  6,  6,  8, 12,  4,  6,  6,  2,  6, 12,
       10, 18,  2,  4,  6,  2,  6,  4,  2,  4, 12,  2,  6, 34,  6,  6,  8,
       18, 10, 14,  4,  2

In [39]:
# comparing average gap sizes for small primes, for the counterexample, and for the cycle G(113#)
print(f"pi avg {(3251/458):.4f} vs mu_Eng {(3242/459):.4f} vs mu_113 {(1/mu_recip):.4f}")

pi avg 7.0983 vs mu_Eng 7.0632 vs mu_113 8.7131


In [40]:
mu_gaps = np.zeros(100)
mu_gaps[0] = 2
i=1
while (i<100):
    mu_gaps[i] = mu_gaps[i-1] * (smallp[i]/(smallp[i]-1))
    i += 1

In [41]:
mu_gaps[0:20]

array([2.        , 3.        , 3.75      , 4.375     , 4.8125    ,
       5.21354167, 5.53938802, 5.8471318 , 6.11291052, 6.33122875,
       6.54226971, 6.72399942, 6.89209941, 7.05619701, 7.2095926 ,
       7.34823861, 7.47493238, 7.59951459, 7.71465875, 7.82486816])

In [42]:
smallp[444:470]

array([3121, 3137, 3163, 3167, 3169, 3181, 3187, 3191, 3203, 3209, 3217,
       3221, 3229, 3251, 3253, 3257, 3259, 3271, 3299, 3301, 3307, 3313,
       3319, 3323, 3329, 3331])

In [43]:
smallp[0:5]

array([ 2,  3,  5,  7, 11])

In [44]:
# searching for instances that could survive the sieve.  Far too many copies to search exhaustively, so we pursue a greedy random
# algorithm.  Search randomly for an mk=0, then search greedily for additional 0's
nprimes = len(smallp)
print(f" nprimes {nprimes} maxp {smallp[nprimes-1]}")

 nprimes 646030 maxp 9699691


In [45]:
# random probes through the admissible extensions from the prefixes in gammam_list
# 
k0 = len(gammam_list[0])  # first open index for pk.  Remember the shift pk=smallp[k+4].  smallp[4]=11
k1 = len(rezlist)
print(f"k {k0}-{k1} pmax {smallp[k1+4]}")
logpml = np.log10(2310)
i=5
while (i < (k1+4)):
    logpml = logpml + np.log10(smallp[i])
    i += 1
print(f"i {i-1} p {smallp[i-1]} log(pml) {logpml}")

k 37-665 pmax 5003
i 668 p 4999 log(pml) 2133.1221880361404


In [46]:
# XXXQHERE [21 May] -- random extensions
# opportunistic search:  random search until we find an mk=0, then search for adjacent mk=0
ntries = 2 # 000000
itry = 0
while (itry < ntries):
    iprefix = random.randrange(len(gammam_list))
    test_gamma = gammam_list[iprefix].copy()
    # randomly extend this prefix until we find an m=0

    # try to extend with another 0

    itry += 1

In [ ]:
# Repeating the analysis for case (458,3240)


In [50]:
# For the primorial coordinates for the counterexample we start in G(11#), so we start with the unique driving term in that cycle
Engelsma458_11 = np.array([2,4,2,4,6,2,6,4,2,4,6,6,2,6,6,6,4,6,8,4,2,4,2,4,8,6,4,8,4,6,2,6,6,4,2,4,6,8,4,2,4,2,10,2,10,2,4,2,4,6,2,10,2,4,6,8,6,4,2,6,4,6,8,4,6,2,4,
              8,6,4,6,2,4,6,2,6,6,4,6,6,2,6,6,4,2,10,2,10,2,4,2,4,6,2,6,4,2,10,6,2,6,4,2,6,4,6,8,4,2,4,2,12,6,4,6,2,4,6,2,12,4,2,4,8,6,4,2,4,2,10,2,10,6,2,
              4,6,2,6,4,2,4,6,6,2,6,4,2,10,6,8,6,4,2,4,8,6,4,6,2,4,6,2,6,6,6,4,6,2,6,4,2,4,2,10,12,2,4,2,10,2,6,4,2,4,6,6,2,10,2,6,4,14,4,2,4,2,4,8,6,4,
              6,2,4,6,2,6,6,4,2,4,6,2,6,4,2,4,12,2,12,4,2,4,6,2,6,4,2,4,6,6,2,6,4,2,6,4,6,8,4,2,4,2,4,14,4,6,2,10,2,6,6,4,2,4,6,2,10,2,4,2,12,10,2,4,2,
              4,6,2,6,4,6,6,6,2,6,4,2,6,4,6,8,4,2,4,6,8,6,10,2,4,6,2,6,6,4,2,4,6,2,6,4,2,6,10,2,10,2,4,2,4,6,8,4,2,4,12,2,6,4,2,6,4,6,12,2,4,2,4,8,6,4,6,2,
              4,6,2,6,10,2,4,6,2,6,4,2,4,2,10,2,10,2,4,6,6,2,6,6,4,6,6,2,6,4,2,6,4,6,8,4,2,6,4,8,6,4,6,2,4,6,8,6,4,2,10,2,6,4,2,4,2,10,2,10,2,4,2,4,8,6,4,
              2,4,6,6,2,6,4,8,4,6,8,4,2,4,2,4,8,6,4,6,6,6,2,6,6,4,2,4,6,2,6,4,2,4,2,10,2,10,2,6,4,6,2,6,4,2,4,6,6,8,4,2,6,10,8,4,2,4,2,4,8,10,6,2,4,8,6,
              6,4,2,4,6,2,6,4,6,2,10,2,10,2,4,2,4,6,2,6,4,2,4,6,6,2,6,6,6,4,6,8,4,2,4,2,4,8,6,4,8,4,6,2,6,6,4,2,4,6,8,4,2,4,2,10,2,10,2,4,2,4,6,2,10,2,
              4,6,8,6,4,2,6,4,6,8,4,6,2,4,8,6,4,6,2,4,6,2,6,6,4,6,6,2,6,6,4,2,10,2,10,2,4,2,4,6,2,6,4,2,10,6,2,6,4,2,6,4,6,8,4,2,4,2,12,6,4,6,2,4,6,2,
              12,4,2,4,8,6,4,2,4,2,10,2,10,6,2,4,6,2,6,4,2,4,6,6,2,6,4,2,10,6,8,6,4,2,4,8,6,4,6,2,4,6,2,6,6,6,4,6,2,6,4,2,4,2,10,12,2,4,2,10,2,6,4,2,4,6,
              6,2,10,2,6,4,14,4], dtype=int)
print(f"Driving term for Engelsma ({len(Engelsma458_11)},{np.sum(Engelsma458_11)}) counterexample")

Driving term for Engelsma (673,3240) counterexample


In [29]:
smallp[41]

np.int64(181)

In [30]:
smallp[37:42]

array([163, 167, 173, 179, 181])

In [25]:
smallp[450:460]

array([3187, 3191, 3203, 3209, 3217, 3221, 3229, 3251, 3253, 3257])

In [26]:
prime_constellation[0:459]

array([ 2,  1,  2,  2,  4,  2,  4,  2,  4,  6,  2,  6,  4,  2,  4,  6,  6,
        2,  6,  4,  2,  6,  4,  6,  8,  4,  2,  4,  2,  4, 14,  4,  6,  2,
       10,  2,  6,  6,  4,  6,  6,  2, 10,  2,  4,  2, 12, 12,  4,  2,  4,
        6,  2, 10,  6,  6,  6,  2,  6,  4,  2, 10, 14,  4,  2,  4, 14,  6,
       10,  2,  4,  6,  8,  6,  6,  4,  6,  8,  4,  8, 10,  2, 10,  2,  6,
        4,  6,  8,  4,  2,  4, 12,  8,  4,  8,  4,  6, 12,  2, 18,  6, 10,
        6,  6,  2,  6, 10,  6,  6,  2,  6,  6,  4,  2, 12, 10,  2,  4,  6,
        6,  2, 12,  4,  6,  8, 10,  8, 10,  8,  6,  6,  4,  8,  6,  4,  8,
        4, 14, 10, 12,  2, 10,  2,  4,  2, 10, 14,  4,  2,  4, 14,  4,  2,
        4, 20,  4,  8, 10,  8,  4,  6,  6, 14,  4,  6,  6,  8,  6, 12,  4,
        6,  2, 10,  2,  6, 10,  2, 10,  2,  6, 18,  4,  2,  4,  6,  6,  8,
        6,  6, 22,  2, 10,  8, 10,  6,  6,  8, 12,  4,  6,  6,  2,  6, 12,
       10, 18,  2,  4,  6,  2,  6,  4,  2,  4, 12,  2,  6, 34,  6,  6,  8,
       18, 10, 14,  4,  2

In [51]:
smallp[50:89]

array([233, 239, 241, 251, 257, 263, 269, 271, 277, 281, 283, 293, 307,
       311, 313, 317, 331, 337, 347, 349, 353, 359, 367, 373, 379, 383,
       389, 397, 401, 409, 419, 421, 431, 433, 439, 443, 449, 457, 461])

In [54]:
i = 1
w=1.0
primpk = np.ones(200, dtype=float)
primpk[0] = 2.0
while (i < 130):
    # w *= (smallp[i]-1)
    primpk[i] = smallp[i] * primpk[i-1]
    print(f"{i:3d} p {smallp[i]:4d} p# {primpk[i]}")
    i +=1
# switch to log-primorial
primpk[i]= np.log10(primpk[i-1]) + np.log10(smallp[i])
print(f"{i:3d} p {smallp[i]:4d} log-p# {primpk[i]}")
i += 1
while (i < 200):
    primpk[i]= primpk[i-1] + np.log10(smallp[i])
    print(f"{i:3d} p {smallp[i]:4d} log-p# {primpk[i]}")
    i += 1
    

  1 p    3 p# 6.0
  2 p    5 p# 30.0
  3 p    7 p# 210.0
  4 p   11 p# 2310.0
  5 p   13 p# 30030.0
  6 p   17 p# 510510.0
  7 p   19 p# 9699690.0
  8 p   23 p# 223092870.0
  9 p   29 p# 6469693230.0
 10 p   31 p# 200560490130.0
 11 p   37 p# 7420738134810.0
 12 p   41 p# 304250263527210.0
 13 p   43 p# 1.308276133167003e+16
 14 p   47 p# 6.148897825884914e+17
 15 p   53 p# 3.2589158477190046e+19
 16 p   59 p# 1.9227603501542128e+21
 17 p   61 p# 1.1728838135940697e+23
 18 p   67 p# 7.858321551080267e+24
 19 p   71 p# 5.57940830126699e+26
 20 p   73 p# 4.072968059924903e+28
 21 p   79 p# 3.2176447673406735e+30
 22 p   83 p# 2.670645156892759e+32
 23 p   89 p# 2.3768741896345556e+34
 24 p   97 p# 2.3055679639455188e+36
 25 p  101 p# 2.328623643584974e+38
 26 p  103 p# 2.398482352892523e+40
 27 p  107 p# 2.5663761175949998e+42
 28 p  109 p# 2.79734996817855e+44
 29 p  113 p# 3.1610054640417614e+46
 30 p  127 p# 4.014476939333037e+48
 31 p  131 p# 5.258964790526278e+50
 32 p  137 p# 7.204

In [66]:
smallp[-1]

np.int64(9699691)